In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

usual = pd.read_csv("nli_label_counts.csv")
mini = pd.read_csv("mini_nli_label_counts.csv")

usual["model"] = "обычная"
mini["model"] = "мини"

usual["model_full"] = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
mini["model_full"] = "MoritzLaurer/multilingual-MiniLMv2-L12-mnli-xnli"

df = pd.concat([usual, mini], ignore_index=True)

methods = ["simple", "decomposed", "threshold"]
models = ["обычная", "мини"]
labels = ["entailment", "neutral", "contradiction"]

label_names = {
    "entailment": "Entailment",
    "neutral": "Neutral",
    "contradiction": "Contradiction",
}

model_names = {
    "обычная": "mDeBERTa-v3-base",
    "мини": "MiniLMv2-L12",
}

colors = {
    "contradiction": "red",
}

rows = []

for model in models:
    for method in methods:
        part = df[(df["model"] == model) & (df["method"] == method)]

        row = {
            "name": f"{model_names[model]}, {method}"
        }

        for label in labels:
            value = part[part["label"] == label]["rate_pct"].iloc[0]
            row[label] = value

        rows.append(row)

plot_df = pd.DataFrame(rows)

y = np.arange(len(plot_df))
height = 0.22

fig, ax = plt.subplots(figsize=(10, 5))

for i, label in enumerate(labels):
    values = plot_df[label].values
    shift = (i - 1) * height

    bars = ax.barh(
        y + shift,
        values,
        height,
        label=label_names[label],
        color=colors.get(label),
    )

    for bar, value in zip(bars, values):
        ax.text(
            value + 1,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.0f}%",
            va="center",
            fontsize=9,
        )

ax.set_yticks(y)
ax.set_yticklabels(plot_df["name"])
ax.set_xlabel("Доля примеров, %")
ax.set_ylabel("Модель и метод")
ax.set_xlim(0, 100)
ax.legend(title="Класс", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("nli_label_counts_horizontal.png", dpi=200, bbox_inches="tight")
plt.show()